# Híbridos SSM-Transformer — Gate da Fase A (Colab)

Runtime → **GPU** (T4/L4/A100). Execute as células em ordem. A Fase A é um
**gate**: só avance para B/C se tudo aqui passar sem `nan`.

Backend: `setup_env.py` instala os kernels Mamba-2 por wheel pré-compilada ou
cai para o backend PyTorch puro (`transformers.Mamba2`). Tudo roda **inline**
(sem `!python`), pois o backend não sobrevive a subprocesso.

## 1 — GPU e Google Drive (checkpoints sobrevivem a quedas)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('bf16 suportado:', torch.cuda.is_bf16_supported())

from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = '/content/drive/MyDrive/hybrid_ckpts'
import os; os.makedirs(OUT_ROOT, exist_ok=True)
print('checkpoints em:', OUT_ROOT)

## 2 — Clonar o repositório

In [ ]:
import os
REPO_URL = 'https://github.com/Mavitu56/Hibridos-em-LLMs-Tranformers-SSMs.git'
REPO_DIR = '/content/hybrid-ssm-lm'
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
os.chdir(REPO_DIR)
import sys; sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())

## 3 — Dependências base

`--only-binary=:all:` força o pip a usar **somente wheels pré-compiladas** — se
faltar wheel para o Python do Colab, ele falha alto com 'no binary' em vez de
cair para build do source (o erro `Getting requirements to build wheel`).
**Não reinstalamos o torch** (o do Colab já casa com a CUDA do runtime).

In [ ]:
import sys
print('Python do Colab:', sys.version.split()[0])
# Instala só por wheel. Se algum pacote não tiver wheel, o comando falha e diz
# qual — bem mais legível que um build de Rust/C++ no meio do caminho.
!pip install -q --only-binary=:all: -r requirements.txt
print('\n✓ dependências base instaladas (somente wheels)')

## 4 — Backend Mamba-2 (kernels por wheel OU fallback torch)
Define `MAMBA_BACKEND`. Se a wheel não casar, cai para torch puro **sem compilar**.

In [ ]:
import setup_env
BACKEND = setup_env.setup()
print('Backend ativo:', BACKEND)

## 5 — [A2] Smoke de blocos (forward sem nan, shape ok)

In [ ]:
import run_experiments as R
assert R.smoke_blocks(), 'smoke de blocos falhou'

## 6 — [A3] Paridade de parâmetros (±5% de 50M ativos)

In [ ]:
import check_parity
assert check_parity.check_all_variants(), 'paridade fora de banda — ajuste d_ff_mamba'

## 7 — Selftests dos benchmarks sintéticos (MQAR e RULER)

In [ ]:
from eval import mqar, ruler
assert mqar.selftest(), 'selftest MQAR falhou'
assert ruler.selftest(), 'selftest RULER falhou'

## 8 — Smoke do dataloader (sem baixar tudo)

In [ ]:
from config import ModelConfig, TrainConfig
from data.dataloader import make_dataloader
tc = TrainConfig(batch_size=2, block_size=256, max_tokens=256*20)
loader = make_dataloader(tc, ModelConfig(), split='train', num_workers=0)
x, y = next(iter(loader))
print('x', tuple(x.shape), 'y', tuple(y.shape))
assert (x[0,1:] == y[0,:-1]).all(), 'targets não estão shifted by 1'
print('✓ dataloader OK')

## 9 — [A4] Smoke train (~50 steps) + resume
Loss deve cair, sem `nan`; o resume retoma do checkpoint.

In [ ]:
assert R.smoke_train(OUT_ROOT), 'smoke train falhou'

## 10 — [A5] Baselines até o orçamento (attn_only e ssm_only)
Pode levar horas. Os checkpoints vão para o Drive; se a sessão cair, reexecute
esta célula — o treino retoma do último checkpoint.

In [ ]:
from train import train
from config import TrainConfig
for v in ('attn_only', 'ssm_only'):
    print('==== baseline', v, '====')
    train(v, TrainConfig(out_dir=f'{OUT_ROOT}/{v}'))

## 11 — Fase B (núcleo da hipótese): hybrid_3_1 + MQAR/ppl
Só rode após a Fase A passar inteira.

In [ ]:
R.phase_b(OUT_ROOT)